In [2]:
!python3 -m pip install langchain_community
import os
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

  Using cached mypy_extensions-1.1.0-py3-none-any.whl.metadata (1.1 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 22.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 28.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 5.8 MB/s  0:00:00 eta 0:00:01
Using cached mypy_extensions-1.1.0-py3-none-any.whl (5.0 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18/18 [langchain_community]ngchain_community]

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip3 install --upgrade pip


In [4]:
!python3 -m pip install python-dotenv
from dotenv import load_dotenv

load_dotenv()


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip3 install --upgrade pip


True

In [5]:
# 1. Setup the model

llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0.0)

In [6]:
# 2. Setup the Prompt with a MessagesPlaceholder for the history

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful AI Assitant"),
    MessagesPlaceholder(variable_name="chat_history"), # This is exactly where past messages get injected!
    ("human", "{question}")
])

In [7]:
# 3. Creating basic LCEL Chain

chain = prompt | llm

In [9]:
# 4. Create a mock database dictionary to store histories for different users

store = {} # Local Python memory

# stores memony like this --> In-memory session storage
# store = {
#   "user_123": [
#     Human: "Hi, my favorite anime is Demon Slayer",
#     AI: "Demon Slayer is a popular anime..."
#   ]
# }

def get_session_history(session_id: str):
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

In [10]:
# 5. THE MAGIC: Wrap the chain with the modern History manager

memory_chain = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="question",
    history_messages_key="chat_history"
)

In [11]:
# 6. Invoke it! Notice we must pass a session_id to track who is talking

response1 = memory_chain.invoke(
    {"question": "Hi, my favorite anime is Demon Slayer."},
    config={"configurable": {"session_id": "user_123"}}
)
print("Turn 1:", response1.content)

response2 = memory_chain.invoke(
    {"question": "What did I say my favorite anime was?"},
    config={"configurable": {"session_id": "user_123"}}
)
# Turn 2 will successfully remember Demon Slayer!
print("Turn 2:", response2.content)

Turn 1: Demon Slayer (Kimetsu no Yaiba) is an extremely popular and highly acclaimed anime series. It's known for its stunning animation, engaging storyline, and well-developed characters. The series follows the story of Tanjiro Kamado, a young boy who becomes a demon slayer after his family is slaughtered by demons, and his sister Nezuko is turned into a demon.

What's your favorite aspect of Demon Slayer? Is it the action scenes, the character development, or something else?
Turn 2: You said your favorite anime is Demon Slayer (Kimetsu no Yaiba).
